In [ ]:
import mlflow
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

spark = SparkSession.builder.getOrCreate()

mlflow.set_registry_uri("databricks-uc")

model_uri = "models:/northmart_dev.ml.fraud_detection_model/1"

model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/northmart_dev/ml/mlflow_tmp"
)

features = (
    spark.table("northmart_dev.silver.fraud_features_5min")
    .select(
        "window",
        "card_id",
        "transaction_count_5min",
        "amount_sum_5min",
        "is_fraud"
    )
)

predictions = model.transform(features)

result = (
    predictions
    .select(
        "window",
        "card_id",
        "transaction_count_5min",
        "amount_sum_5min",
        "is_fraud",
        "prediction",
        "probability"
    )
    .withColumn("scored_at", current_timestamp())
)

result.write.mode("overwrite").saveAsTable(
    "northmart_dev.gold.fraud_predictions"
)

2026/08/22 11:45:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/22 11:45:57 INFO mlflow.store.db.utils: Updating database tables
2026/08/22 11:46:12 INFO mlflow.spark: URI 'models:/northmart_dev.ml.fraud_detection_model/1/sparkml' does not point to the current DFS.
2026/08/22 11:46:12 INFO mlflow.spark: File 'models:/northmart_dev.ml.fraud_detection_model/1/sparkml' not found on DFS. Will attempt to upload the file.


AttributeError: 'NoneType' object has no attribute 'jvm'

In [2]:
dbutils.secrets.get(scope = "kv-northmart-gmkng", key = "northmart-sql-user")

'northmartadmin'